Data Cleaning where data having anomalies

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("C:/Users/admin/Downloads/jet_speed_real.xlsx", header=8)
df.head()

,PROJECT_NAME,TIMESTAMP,PM2.5,PM10,TEMPERATURE,HUMIDITY
0,Jet Speed Real Tors,2025-12-01T00:00:24+05:30,85.0,141.0,24.6,61.6
1,Jet Speed Real Tors,2025-12-01T00:00:54+05:30,84.0,145.0,24.6,61.4
2,Jet Speed Real Tors,2025-12-01T00:01:24+05:30,85.0,145.0,24.5,61.3
3,Jet Speed Real Tors,2025-12-01T00:01:54+05:30,80.0,141.0,24.5,61.8
4,Jet Speed Real Tors,2025-12-01T00:02:24+05:30,88.0,143.0,24.6,62.4


In [2]:
#Converting from timestamp to datetime
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], format='mixed')

In [3]:
#Set Index
df = df.set_index('TIMESTAMP')

In [4]:
# Sort properly
df = df.sort_values('TIMESTAMP')

In [5]:
# Step 4: Remove unrealistic values

df_clean = df.copy()

# Set extreme values to NaN
df_clean.loc[df_clean['PM2.5'] > 500, 'PM2.5'] = np.nan
df_clean.loc[df_clean['PM10'] > 600, 'PM10'] = np.nan

df_clean[['PM2.5','PM10']].describe()

,PM2.5,PM10
count,209640.000000,209592.000000
mean,73.478045,126.333225
std,20.417491,41.335691
min,6.000000,14.000000
25%,59.000000,97.000000
50%,72.000000,123.000000
75%,86.000000,150.000000
max,322.000000,600.000000


In [6]:
# Step 5: Fill missing values smoothly

df_clean['PM2.5'] = df_clean['PM2.5'].interpolate(method='time')
df_clean['PM10'] = df_clean['PM10'].interpolate(method='time')

In [7]:
# Step 6: Apply rolling mean smoothing

df_clean['PM2.5'] = df_clean['PM2.5'].rolling(window=3, min_periods=1).mean()
df_clean['PM10'] = df_clean['PM10'].rolling(window=3, min_periods=1).mean()

In [8]:
# Step 7: Clip to safe realistic ranges

df_clean['PM2.5'] = df_clean['PM2.5'].clip(0, 300)
df_clean['PM10'] = df_clean['PM10'].clip(0, 400)

In [10]:
#Round-Of Data
df_clean[['PM2.5','PM10','TEMPERATURE','HUMIDITY']] = df_clean[['PM2.5','PM10','TEMPERATURE','HUMIDITY']].round(2)

In [12]:
df_clean

,PROJECT_NAME,PM2.5,PM10,TEMPERATURE,HUMIDITY
TIMESTAMP,,,,,
2025-12-01 00:00:24+05:30,Jet Speed Real Tors,85.00,141.00,24.6,61.6
2025-12-01 00:00:54+05:30,Jet Speed Real Tors,84.50,143.00,24.6,61.4
2025-12-01 00:01:24+05:30,Jet Speed Real Tors,84.67,143.67,24.5,61.3
2025-12-01 00:01:54+05:30,Jet Speed Real Tors,83.00,143.67,24.5,61.8
2025-12-01 00:02:24+05:30,Jet Speed Real Tors,84.33,143.00,24.6,62.4
...,...,...,...,...,...
2026-03-25 10:45:14+05:30,Jet Speed Real Tors,82.31,82.31,38.3,50.3
2026-03-25 11:00:18+05:30,Jet Speed Real Tors,81.77,81.77,38.6,51.5
2026-03-25 11:15:22+05:30,Jet Speed Real Tors,81.18,81.18,39.0,50.8


In [13]:
# Step 10: Save cleaned dataset

df_clean.to_csv("cleaned_data.csv")

print("✅ Clean data saved successfully!")

✅ Clean data saved successfully!


In [2]:
import pandas as pd
df = pd.read_excel("C:/Users/admin/Downloads/jet_speed_26march.xlsx", header= 8)
df.head()

,PROJECT_NAME,TIMESTAMP,PM2.5,PM10,TEMPERATURE,HUMIDITY
0,Jet Speed Real Tors,2026-03-26T00:06:34+05:30,82.900,82.900,29.2,82.6
1,Jet Speed Real Tors,2026-03-26T00:21:47+05:30,83.575,83.575,29.2,83.4
2,Jet Speed Real Tors,2026-03-26T00:37:16+05:30,83.725,83.725,29.1,83.4
3,Jet Speed Real Tors,2026-03-26T00:52:49+05:30,84.825,84.825,29.0,83.5
4,Jet Speed Real Tors,2026-03-26T01:07:54+05:30,83.450,83.450,28.8,83.7


In [3]:
#Converting from timestamp to datetime
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], format='mixed')

In [4]:
#Set Index
df = df.set_index('TIMESTAMP')

In [5]:
#Resample data into 1hour batch
sensor_df = df.resample('1h').mean(numeric_only=True)

In [6]:
#Checking  Type of Index
print(type(df.index))

<class 'pandas.core.indexes.datetimes.DatetimeIndex'>


In [7]:
#Interpolate: to replace with mode value of any  columns
sensor_df = sensor_df.interpolate(limit_direction='forward')

In [8]:
#Round-Of Data
sensor_df[['PM2.5','PM10','TEMPERATURE','HUMIDITY']] = sensor_df[['PM2.5','PM10','TEMPERATURE','HUMIDITY']].round(2)

In [9]:
#Reset Index
sensor_df = sensor_df.reset_index()

In [12]:
sensor_df.head(20)

,TIMESTAMP,PM2.5,PM10,TEMPERATURE,HUMIDITY
0,2026-03-26 00:00:00+05:30,83.76,83.76,29.12,83.22
1,2026-03-26 01:00:00+05:30,83.96,83.96,28.65,83.85
2,2026-03-26 02:00:00+05:30,83.48,83.48,28.30,83.85
3,2026-03-26 03:00:00+05:30,83.28,83.28,27.92,83.72
4,2026-03-26 04:00:00+05:30,83.41,83.41,27.65,85.82
5,2026-03-26 05:00:00+05:30,83.11,83.11,27.35,83.68
6,2026-03-26 06:00:00+05:30,83.21,83.21,27.12,80.65
7,2026-03-26 07:00:00+05:30,83.32,83.32,28.00,77.45
8,2026-03-26 08:00:00+05:30,83.25,83.25,30.32,63.28
9,2026-03-26 09:00:00+05:30,83.55,83.55,33.62,49.28
